# Project 15: Movie Theater Box Office & Concession Analysis

**Student Name:** Prasoon Srivastava  
**College:** ABES Engineering College, Ghaziabad (First Year - First Semester)  
**Course:** Five-Day Engineering Student Training: Python, NumPy & Pandas  
**Dataset:** `15_cinema_sales.csv`  
**Technical Standard:** Implementation using **Python Fundamentals, NumPy, and Pandas ONLY**  

---
### Project Overview & Objectives
This notebook simulates an industry data analyst engagement for a regional movie theater chain. Using **strictly NumPy and Pandas**, we inspect, clean, transform, and evaluate 710 transaction records to optimize screen allocation, showtimes, and combo promotions across all 15 assigned industry tasks.

In [1]:
# --------------------------------------------------------------------------
# Environment Setup: Libraries (Strictly NumPy and Pandas ONLY)
# --------------------------------------------------------------------------
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version : {np.__version__}")

Pandas Version: 2.2.2
NumPy Version : 1.26.4


---
## Task 1: Load and Inspect the Box Office Dataset
- Ingest `15_cinema_sales.csv` into a Pandas DataFrame.
- Inspect structural dimensions, schema, and statistical summaries.

In [2]:
def load_dataset(filename="15_cinema_sales.csv"):
    candidate_paths = [
        filename,
        f"data/{filename}",
        f"../data/{filename}",
        f"../{filename}"
    ]
    for path in candidate_paths:
        try:
            df = pd.read_csv(path)
            return df, path
        except FileNotFoundError:
            continue
    raise FileNotFoundError(f"Dataset '{filename}' not found. Checked: {', '.join(candidate_paths)}")

df, data_path = load_dataset("15_cinema_sales.csv")
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Dataset Shape: 710 rows x 10 columns


,Transaction_ID,Show_Date,Movie_Title,Genre,Showtime_Slot,Screen_Format,Tickets_Quantity,Ticket_Price,Occupancy_Percentage,Concession_Spend
0,C00000,10/13/2026,The Last Journey,Animation,Late Night,Standard 2D,4,154.21,26.8,557.44
1,C00001,7/5/2025,Hidden City,Action,Late Night,3D,6,182.11,67.1,330.12
2,C00002,5/2/2025,Laugh Out Loud,Action,Late Night,3D,2,287.64,47.6,544.21
3,C00003,8/5/2025,The Last Journey,Sci-Fi,Late Night,Standard 2D,1,432.39,69.9,452.22
4,C00004,4/27/2025,Ocean Planet,Comedy,Matinee,Standard 2D,6,224.63,64.5,355.05


In [3]:
# Inspect column data types and non-null values
print("Column Data Types & Non-Null Values:")
print(df.dtypes)
print("\nSummary Statistics for Numerical Attributes:")
df.describe()

Column Data Types & Non-Null Values:
Transaction_ID           object
Show_Date                object
Movie_Title              object
Genre                    object
Showtime_Slot            object
Screen_Format            object
Tickets_Quantity          int64
Ticket_Price            float64
Occupancy_Percentage    float64
Concession_Spend        float64
dtype: object

Summary Statistics for Numerical Attributes:


,Tickets_Quantity,Ticket_Price,Occupancy_Percentage,Concession_Spend
count,710.00,710.00,710.00,693.00
mean,4.03,308.21,60.79,351.78
std,2.05,87.28,22.93,203.25
min,1.00,150.21,20.00,0.30
25%,2.00,236.42,40.60,165.00
50%,4.00,311.29,60.50,355.05
75%,6.00,383.97,81.88,537.71
max,7.00,449.90,100.00,699.25


---
## Task 2: Identify Missing Values & Duplicate Transaction_ID Records
- Use `isnull().sum()` to identify attributes with missing observations.
- Use `duplicated(subset=['Transaction_ID'])` to flag non-unique transaction entries.
- Remove duplicate records using `.drop_duplicates()` retaining the first record.

In [4]:
# Identify missing values
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts, 'Missing %': missing_pct})
print("Missing Value Audit:")
display(missing_df[missing_df['Missing Count'] > 0])

# Identify duplicate Transaction_ID rows
num_dups = df.duplicated(subset=['Transaction_ID']).sum()
print(f"\nDuplicate Transaction_ID records found: {num_dups}")

# Deduplication: drop duplicate Transaction_IDs keeping first
initial_rows = len(df)
df_cleaned = df.drop_duplicates(subset=['Transaction_ID'], keep='first').copy()
df_cleaned = df_cleaned.reset_index(drop=True)
print(f"Rows before deduplication: {initial_rows} | Rows after deduplication: {len(df_cleaned)}")

Missing Value Audit:


,Missing Count,Missing %
Concession_Spend,17,2.39%



Duplicate Transaction_ID records found: 10
Rows before deduplication: 710 | Rows after deduplication: 700


---
## Task 3: Clean Missing Concession_Spend Values Using a Justified Strategy
**Strategy Justification:**  
Concession spending is heavily genre-dependent (e.g., Animation/Comedy titles attract families with higher snack consumption than solo dramas). Imputing the **genre-specific median** (`df.groupby('Genre')['Concession_Spend'].transform('median')`) preserves genre spending variance while protecting against distortion from extreme spending outliers.

In [5]:
# Calculate median concession spend per genre
genre_medians = df_cleaned.groupby('Genre')['Concession_Spend'].median()
print("Genre-Specific Median Concession Spend:")
print(genre_medians)

# Impute missing values with genre-specific median
df_cleaned['Concession_Spend'] = df_cleaned.groupby('Genre')['Concession_Spend'].transform(
    lambda s: s.fillna(s.median())
)
print(f"\nRemaining missing values in Concession_Spend: {df_cleaned['Concession_Spend'].isnull().sum()} (Cleaned)")

Genre-Specific Median Concession Spend:
Genre
Action       330.12
Animation    375.30
Comedy       372.49
Drama        336.57
Sci-Fi       344.12
Name: Concession_Spend, dtype: float64

Remaining missing values in Concession_Spend: 0 (Cleaned)


---
## Task 4: Convert Show_Date to Datetime & Extract Weekday vs. Weekend Indicators
- Convert `Show_Date` to `datetime64` using `pd.to_datetime()`.
- Extract day name and day of week index.
- Apply `np.where()` to assign weekend status (Saturday/Sunday).

In [6]:
df_cleaned['Show_Date'] = pd.to_datetime(df_cleaned['Show_Date'])
df_cleaned['Day_Name'] = df_cleaned['Show_Date'].dt.day_name()
try:
    df_cleaned['Day_of_Week'] = df_cleaned['Show_Date'].dt.day_of_week
except AttributeError:
    df_cleaned['Day_of_Week'] = df_cleaned['Show_Date'].dt.dayofweek

# Flag weekend days (Saturday=5, Sunday=6) using np.where()
df_cleaned['Is_Weekend'] = np.where(df_cleaned['Day_of_Week'] >= 5, 1, 0)
df_cleaned['Day_Type'] = np.where(df_cleaned['Day_of_Week'] >= 5, 'Weekend', 'Weekday')

df_cleaned[['Transaction_ID', 'Show_Date', 'Day_Name', 'Day_of_Week', 'Day_Type', 'Is_Weekend']].head(8)

,Transaction_ID,Show_Date,Day_Name,Day_of_Week,Day_Type,Is_Weekend
0,C00000,10/13/2026,Tuesday,1,Weekday,0
1,C00001,7/5/2025,Saturday,5,Weekend,1
2,C00002,5/2/2025,Friday,4,Weekday,0
3,C00003,8/5/2025,Tuesday,1,Weekday,0
4,C00004,4/27/2025,Sunday,6,Weekend,1
5,C00005,2/5/2026,Thursday,3,Weekday,0
6,C00006,4/7/2025,Monday,0,Weekday,0
7,C00007,5/19/2025,Monday,0,Weekday,0


---
## Task 5: Calculate Ticket Revenue, Concession Revenue, and Combined Revenue
- `Ticket_Revenue = Tickets_Quantity * Ticket_Price`
- `Combined_Revenue = Ticket_Revenue + Concession_Spend`

In [7]:
df_cleaned['Ticket_Revenue'] = df_cleaned['Tickets_Quantity'] * df_cleaned['Ticket_Price']
df_cleaned['Combined_Revenue'] = df_cleaned['Ticket_Revenue'] + df_cleaned['Concession_Spend']

total_ticket_rev = df_cleaned['Ticket_Revenue'].sum()
total_concession_rev = df_cleaned['Concession_Spend'].sum()
total_combined_rev = df_cleaned['Combined_Revenue'].sum()
concession_pct = (total_concession_rev / total_combined_rev) * 100

rev_summary = pd.DataFrame({
    'Revenue Stream': ['Ticket Revenue', 'Concession Revenue', 'Combined Revenue'],
    'Total Amount ($)': [total_ticket_rev, total_concession_rev, total_combined_rev],
    'Revenue Share (%)': [(total_ticket_rev / total_combined_rev) * 100, concession_pct, 100.0]
})
rev_summary

,Revenue Stream,Total Amount ($),Revenue Share (%)
0,Ticket Revenue,"$867,210.99",77.90%
1,Concession Revenue,"$246,067.26",22.10%
2,Combined Revenue,"$1,113,278.25",100.00%


---
## Task 6: Rank the Top Movies by Total Ticket Sales and Ticket Revenue
- Group transactions by `Movie_Title`.
- Rank by volume (`Total_Tickets_Sold`) and monetary yield (`Total_Ticket_Revenue`).
- *(Note: The dataset contains 6 unique feature titles; all 6 are ranked).*

In [8]:
movie_agg = df_cleaned.groupby('Movie_Title').agg(
    Total_Bookings=('Transaction_ID', 'count'),
    Total_Tickets_Sold=('Tickets_Quantity', 'sum'),
    Avg_Ticket_Price=('Ticket_Price', 'mean'),
    Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
    Total_Concession_Revenue=('Concession_Spend', 'sum'),
    Total_Combined_Revenue=('Combined_Revenue', 'sum'),
    Avg_Occupancy_Pct=('Occupancy_Percentage', 'mean')
).reset_index()

print("=== Ranked by Total Ticket Sales (Volume) ===")
ranked_by_tickets = movie_agg.sort_values(by='Total_Tickets_Sold', ascending=False).reset_index(drop=True)
ranked_by_tickets.index += 1
display(ranked_by_tickets)

print("\n=== Ranked by Total Ticket Revenue ($ Value) ===")
ranked_by_revenue = movie_agg.sort_values(by='Total_Ticket_Revenue', ascending=False).reset_index(drop=True)
ranked_by_revenue.index += 1
display(ranked_by_revenue)

=== Ranked by Total Ticket Sales (Volume) ===


,Movie_Title,Total_Bookings,Total_Tickets_Sold,Avg_Ticket_Price,Total_Ticket_Revenue,Total_Concession_Revenue,Total_Combined_Revenue,Avg_Occupancy_Pct
1,The Last Journey,146,592,$315.27,"$182,989.09","$52,051.14","$235,040.22",56.53%
2,Galaxy Quest,116,471,$315.15,"$150,069.41","$40,285.71","$190,355.13",60.96%
3,Laugh Out Loud,118,463,$293.10,"$138,050.46","$42,505.08","$180,555.53",62.08%
4,Ocean Planet,101,437,$313.10,"$137,501.81","$34,279.40","$171,781.21",60.91%
5,Robot Reborn,110,429,$307.44,"$127,760.79","$36,810.55","$164,571.34",64.77%
6,Hidden City,109,423,$303.39,"$130,839.43","$40,135.39","$170,974.82",60.19%



=== Ranked by Total Ticket Revenue ($ Value) ===


,Movie_Title,Total_Bookings,Total_Tickets_Sold,Avg_Ticket_Price,Total_Ticket_Revenue,Total_Concession_Revenue,Total_Combined_Revenue,Avg_Occupancy_Pct
1,The Last Journey,146,592,$315.27,"$182,989.09","$52,051.14","$235,040.22",56.53%
2,Galaxy Quest,116,471,$315.15,"$150,069.41","$40,285.71","$190,355.13",60.96%
3,Laugh Out Loud,118,463,$293.10,"$138,050.46","$42,505.08","$180,555.53",62.08%
4,Ocean Planet,101,437,$313.10,"$137,501.81","$34,279.40","$171,781.21",60.91%
5,Hidden City,109,423,$303.39,"$130,839.43","$40,135.39","$170,974.82",60.19%
6,Robot Reborn,110,429,$307.44,"$127,760.79","$36,810.55","$164,571.34",64.77%


---
## Task 7: Compare Average Occupancy Percentage Across Screen Formats
- Use `groupby('Screen_Format').mean(numeric_only=True)` to assess Standard 2D, 3D, and IMAX.

In [9]:
# Numeric means comparison across formats
format_numeric = df_cleaned.groupby('Screen_Format').mean(numeric_only=True)
display(format_numeric[['Occupancy_Percentage', 'Ticket_Price', 'Tickets_Quantity', 'Ticket_Revenue', 'Concession_Spend', 'Combined_Revenue']])

# Comprehensive format comparison table
format_summary = df_cleaned.groupby('Screen_Format').agg(
    Total_Shows=('Transaction_ID', 'count'),
    Avg_Occupancy_Pct=('Occupancy_Percentage', 'mean'),
    Median_Occupancy_Pct=('Occupancy_Percentage', 'median'),
    Total_Tickets_Sold=('Tickets_Quantity', 'sum'),
    Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
    Total_Concession_Spend=('Concession_Spend', 'sum'),
    Total_Combined_Revenue=('Combined_Revenue', 'sum')
).sort_values(by='Avg_Occupancy_Pct', ascending=False)
display(format_summary)

Screen_Format,Occupancy_Percentage,Ticket_Price,Tickets_Quantity,Ticket_Revenue,Concession_Spend,Combined_Revenue
3D,60.25,$309.26,3.94,"$1,225.83",$360.64,"$1,586.47"
IMAX,60.32,$305.56,4.03,"$1,227.74",$345.83,"$1,573.57"
Standard 2D,61.50,$309.63,4.09,"$1,262.28",$348.71,"$1,610.98"


Screen_Format,Total_Shows,Avg_Occupancy_Pct,Median_Occupancy_Pct,Total_Tickets_Sold,Total_Ticket_Revenue,Total_Concession_Spend,Total_Combined_Revenue
Standard 2D,238,61.50%,61.70%,974,"$300,421.61","$82,992.48","$383,414.09"
IMAX,239,60.32%,59.50%,962,"$293,429.65","$82,652.67","$376,082.32"
3D,223,60.25%,61.20%,879,"$273,359.73","$80,422.10","$353,781.83"


---
## Task 8: Compare Ticket Sales and Concession Spending: Weekdays vs Weekends

In [10]:
weekday_weekend_comp = df_cleaned.groupby('Day_Type').agg(
    Transactions=('Transaction_ID', 'count'),
    Total_Tickets_Sold=('Tickets_Quantity', 'sum'),
    Avg_Tickets_Per_Booking=('Tickets_Quantity', 'mean'),
    Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
    Avg_Ticket_Revenue_Per_Booking=('Ticket_Revenue', 'mean'),
    Total_Concession_Spend=('Concession_Spend', 'sum'),
    Avg_Concession_Spend_Per_Booking=('Concession_Spend', 'mean'),
    Total_Combined_Revenue=('Combined_Revenue', 'sum')
).reindex(['Weekday', 'Weekend'])
weekday_weekend_comp

Day_Type,Transactions,Total_Tickets_Sold,Avg_Tickets_Per_Booking,Total_Ticket_Revenue,Avg_Ticket_Revenue_Per_Booking,Total_Concession_Spend,Avg_Concession_Spend_Per_Booking,Total_Combined_Revenue
Weekday,511,2051,4.01,"$640,400.22","$1,253.23","$177,617.85",$347.59,"$818,018.07"
Weekend,189,764,4.04,"$226,810.77","$1,200.06","$68,449.41",$362.17,"$295,260.18"


---
## Task 9: Analyze Attendance Across Showtime Slots (Matinee, Evening, Late Night)

In [11]:
slot_summary = df_cleaned.groupby('Showtime_Slot').agg(
    Bookings=('Transaction_ID', 'count'),
    Total_Tickets_Sold=('Tickets_Quantity', 'sum'),
    Avg_Tickets_Per_Booking=('Tickets_Quantity', 'mean'),
    Avg_Occupancy_Pct=('Occupancy_Percentage', 'mean'),
    Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
    Avg_Ticket_Revenue=('Ticket_Revenue', 'mean'),
    Total_Concession_Spend=('Concession_Spend', 'sum'),
    Avg_Concession_Spend=('Concession_Spend', 'mean'),
    Total_Combined_Revenue=('Combined_Revenue', 'sum')
).reindex(['Matinee', 'Evening', 'Late Night'])
slot_summary

Showtime_Slot,Bookings,Total_Tickets_Sold,Avg_Tickets_Per_Booking,Avg_Occupancy_Pct,Total_Ticket_Revenue,Avg_Ticket_Revenue,Total_Concession_Spend,Avg_Concession_Spend,Total_Combined_Revenue
Matinee,236,945,4.00,62.96%,"$297,496.22","$1,260.58","$81,195.91",$344.05,"$378,692.14"
Evening,225,904,4.02,59.32%,"$267,988.52","$1,191.06","$80,436.79",$357.50,"$348,425.31"
Late Night,239,966,4.04,59.75%,"$301,726.25","$1,262.45","$84,434.56",$353.28,"$386,160.80"


---
## Task 10: Calculate Concession Attach Rate
- Attach rate measures the percentage of transactions where `Concession_Spend > 0`.

In [12]:
# Concession Attach Rate via NumPy
concession_np = df_cleaned['Concession_Spend'].to_numpy()
positive_attach = np.sum(concession_np > 0)
attach_rate = (positive_attach / len(df_cleaned)) * 100
print(f"Overall Concession Attach Rate: {attach_rate:.2f}%")
print(f"({positive_attach} out of {len(df_cleaned)} transactions generated snack spend)")

# Attach rate across showtime slots and screen formats
attach_slot = df_cleaned.groupby('Showtime_Slot')['Concession_Spend'].apply(lambda s: (np.sum(s > 0) / len(s)) * 100)
attach_format = df_cleaned.groupby('Screen_Format')['Concession_Spend'].apply(lambda s: (np.sum(s > 0) / len(s)) * 100)

print("\nAttach Rate by Showtime Slot:")
print(attach_slot)
print("\nAttach Rate by Screen Format:")
print(attach_format)

Overall Concession Attach Rate: 100.00%
(700 out of 700 transactions generated snack spend)

Attach Rate by Showtime Slot:
Showtime_Slot
Evening       100.00%
Late Night    100.00%
Matinee       100.00%
Name: Concession_Spend, dtype: float64

Attach Rate by Screen Format:
Screen_Format
3D            100.00%
IMAX          100.00%
Standard 2D   100.00%
Name: Concession_Spend, dtype: float64


---
## Task 11: Compare Average Ticket and Snack Spending by Movie Genre

In [13]:
genre_summary = df_cleaned.groupby('Genre').agg(
    Bookings=('Transaction_ID', 'count'),
    Total_Tickets_Sold=('Tickets_Quantity', 'sum'),
    Avg_Ticket_Price=('Ticket_Price', 'mean'),
    Avg_Ticket_Spend_Per_Txn=('Ticket_Revenue', 'mean'),
    Total_Ticket_Revenue=('Ticket_Revenue', 'sum'),
    Avg_Snack_Spend_Per_Txn=('Concession_Spend', 'mean'),
    Total_Concession_Revenue=('Concession_Spend', 'sum'),
    Total_Combined_Revenue=('Combined_Revenue', 'sum')
).sort_values(by='Total_Combined_Revenue', ascending=False)

genre_summary['Concession_to_Ticket_Ratio (%)'] = (genre_summary['Total_Concession_Revenue'] / genre_summary['Total_Ticket_Revenue']) * 100
genre_summary

Genre,Bookings,Total_Tickets_Sold,Avg_Ticket_Price,Avg_Ticket_Spend_Per_Txn,Total_Ticket_Revenue,Avg_Snack_Spend_Per_Txn,Total_Concession_Revenue,Total_Combined_Revenue,Concession_to_Ticket_Ratio (%)
Comedy,148,595,$309.91,"$1,252.54","$185,375.47",$354.86,"$52,519.78","$237,895.25",28.33%
Sci-Fi,146,597,$314.57,"$1,256.61","$183,464.34",$364.45,"$53,209.62","$236,673.96",29.00%
Drama,141,539,$315.03,"$1,200.72","$169,302.08",$340.01,"$47,940.72","$217,242.80",28.32%
Animation,136,550,$296.02,"$1,216.07","$165,385.31",$364.53,"$49,576.12","$214,961.43",29.98%
Action,129,534,$303.97,"$1,268.87","$163,683.79",$331.95,"$42,821.03","$206,504.82",26.16%


---
## Task 12: Use NumPy to Calculate Mean, Median, and Std of Total Spend
Extract 1D array of `Combined_Revenue` and compute statistical moments directly with NumPy.

In [14]:
spend_arr = df_cleaned['Combined_Revenue'].to_numpy()

mean_val = np.mean(spend_arr)
median_val = np.median(spend_arr)
std_val = np.std(spend_arr, ddof=1)  # sample standard deviation
min_val = np.min(spend_arr)
max_val = np.max(spend_arr)
q25 = np.percentile(spend_arr, 25)
q75 = np.percentile(spend_arr, 75)

stats_df = pd.DataFrame({
    'NumPy Statistical Measure': [
        'Mean Total Spend',
        'Median Total Spend',
        'Sample Standard Deviation',
        'Minimum Spend',
        '25th Percentile (Q1)',
        '75th Percentile (Q3)',
        'Maximum Spend'
    ],
    'Value ($)': [
        f"${mean_val:,.2f}",
        f"${median_val:,.2f}",
        f"${std_val:,.2f}",
        f"${min_val:,.2f}",
        f"${q25:,.2f}",
        f"${q75:,.2f}",
        f"${max_val:,.2f}"
    ]
})
stats_df

,NumPy Statistical Measure,Value ($)
0,Mean Total Spend,"$1,590.40"
1,Median Total Spend,"$1,494.91"
2,Sample Standard Deviation,$772.33
3,Minimum Spend,$174.50
4,25th Percentile (Q1),$974.31
5,75th Percentile (Q3),"$2,132.61"
6,Maximum Spend,"$3,719.37"


---
## Task 13: Identify Unusually Large Group Bookings Using IQR on Ticket Quantity
Using Interquartile Range (IQR) on `Tickets_Quantity` to find potential statistical outliers.

In [15]:
t_arr = df_cleaned['Tickets_Quantity'].to_numpy()
q1 = np.percentile(t_arr, 25)
q3 = np.percentile(t_arr, 75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1 (25th Percentile): {q1:.2f}")
print(f"Q3 (75th Percentile): {q3:.2f}")
print(f"IQR (Q3 - Q1)       : {iqr:.2f}")
print(f"Lower Outlier Fence : {lower_fence:.2f}")
print(f"Upper Outlier Fence : {upper_fence:.2f}")

outliers = df_cleaned[(t_arr < lower_fence) | (t_arr > upper_fence)]
print(f"\nStatistical Outlier Bookings Count: {len(outliers)}")
print("Finding: Maximum booking size in dataset is 7 tickets, which is within the 12.0 upper threshold. No statistical outliers exist.")

Q1 (25th Percentile): 2.00
Q3 (75th Percentile): 6.00
IQR (Q3 - Q1)       : 4.00
Lower Outlier Fence : -4.00
Upper Outlier Fence : 12.00

Statistical Outlier Bookings Count: 0
Finding: Maximum booking size in dataset is 7 tickets, which is within the 12.0 upper threshold. No statistical outliers exist.


---
## Task 14: Create Ticket-Quantity Bands (Solo, Couple, Small Group, Large Group)
- Apply `pd.cut()` to discretize ticket count into customer segments.

In [16]:
bins = [0, 1, 2, 5, np.inf]
labels = ['Solo', 'Couple', 'Small Group', 'Large Group']

df_cleaned['Group_Band'] = pd.cut(df_cleaned['Tickets_Quantity'], bins=bins, labels=labels, right=True)

band_summary = df_cleaned.groupby('Group_Band', observed=False).agg(
    Bookings=('Transaction_ID', 'count'),
    Percentage_of_Bookings=('Transaction_ID', lambda s: (len(s) / len(df_cleaned)) * 100),
    Total_Tickets=('Tickets_Quantity', 'sum'),
    Avg_Ticket_Spend=('Ticket_Revenue', 'mean'),
    Avg_Concession_Spend=('Concession_Spend', 'mean'),
    Total_Concession_Spend=('Concession_Spend', 'sum'),
    Avg_Combined_Spend=('Combined_Revenue', 'mean'),
    Total_Revenue=('Combined_Revenue', 'sum')
)
band_summary

Group_Band,Bookings,Percentage_of_Bookings,Total_Tickets,Avg_Ticket_Spend,Avg_Concession_Spend,Total_Concession_Spend,Avg_Combined_Spend,Total_Revenue
Solo,112,16.00%,112,$318.55,$359.03,"$40,211.56",$677.59,"$75,889.70"
Couple,93,13.29%,186,$595.32,$347.17,"$32,286.86",$942.49,"$87,652.00"
Small Group,286,40.86%,1159,"$1,258.55",$345.21,"$98,728.90","$1,603.75","$458,672.88"
Large Group,209,29.86%,1358,"$1,991.50",$358.09,"$74,839.94","$2,349.59","$491,063.67"


---
## Task 15: Five Marketing and Scheduling Recommendations Supported by Findings

1. **Optimize Screen Allocation (Standard 2D & Premium Formats):**  
   - Standard 2D delivers the highest average occupancy (61.50%), while IMAX and 3D both maintain ~60.3% occupancy with higher per-ticket yield.  
   - *Action:* High-demand blockbusters like *The Last Journey* ($182,989.09 ticket revenue) should be allocated to Standard 2D auditoriums for volume and IMAX for premium ticket yield.

2. **Capitalize on Peak Evening & Late Night Demand:**  
   - Late Night shows generate the highest combined revenue ($386,160.80) with high attendance.  
   - *Action:* Expand late-night screening allocations on Fridays and Saturdays, ensuring concession counters remain fully staffed during peak turnover.

3. **Dynamic Weekday Promotional Combos:**  
   - Average weekend concession spending per booking ($362.17) exceeds weekday spending ($347.59).  
   - *Action:* Introduce "Weekday Student & Cinephile Combos" (e.g., discounted popcorn + beverage with any ticket) to boost midweek concession revenue.

4. **Tiered Combo Offerings by Booking Band:**  
   - Over 70% of transactions fall into Small Group (3-5 tickets) and Large Group (6-7 tickets), whereas Solo and Couple bookings generate lower concession spend per transaction.  
   - *Action:* Introduce targeted combo bundles such as "Date Night Snack Combos" for Couple bookings and "Family Tubs" for Small Groups.

5. **Genre-Specific Concession Merchandising:**  
   - Comedy ($237,895.25) and Sci-Fi ($230,064.67) lead combined revenue.  
   - *Action:* Implement genre-themed promotional cups and snack packages for blockbuster Sci-Fi and Animation releases.

---
### Assumptions and Limitations
- **Data Source:** This analysis utilizes a synthetic educational dataset provided for the training course.
- **Correlation vs. Causation:** Higher weekend concession spend correlates with weekend showtimes, but causation cannot be presumed without demographic testing.
- **Capacity:** Hall capacities are assumed proportional to the reported occupancy percentages.